## Nurislam Aliyev

### Course: Deep Learning

### ID: 24B030024

### Instructor: Senior-Lecturer, Aidyn Abirov

#### Homework 2


#### Imports

In [3]:
import numpy as np
import matplotlib.pyplot as plt

#### Task 1

In [4]:
X = np.array([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
])

y_and = np.array([0, 0, 0, 1])
y_xor = np.array([0, 1, 1, 0])


def predict(X, w, b):
    scores = X @ w + b
    return (scores > 0).astype(int)

def train_perceptron(X, y, eta=0.5, max_epochs=20):

    w = np.zeros(2)
    b = 0.0

    mistakes_per_epoch = []

    for epoch in range(max_epochs):

        mistakes = 0

        for i in range(len(X)):

            y_hat = predict(X[i:i+1], w, b)[0]          # predict BEFORE the update
            if y_hat != y[i]:
                mistakes += 1
            error = y[i] - y_hat                        # 0, +1 or -1
            w = w + eta * error * X[i]
            b = b + eta * error

        mistakes_per_epoch.append(mistakes)

        if mistakes ==0:
            break

    return w, b, mistakes_per_epoch


w_and, b_and, hist_and = train_perceptron(X, y_and)
w_xor, b_xor, hist_xor = train_perceptron(X, y_xor)

print("AND: epochs =", len(hist_and), "w =", w_and, "b =", b_and)
print("AND: mistakes per epoch =", hist_and)
print("AND: final predictions  =", predict(X, w_and, b_and))

print("XOR: epochs =", len(hist_xor), "w =", w_xor, "b =", b_xor)
print("XOR: mistakes per epoch =", hist_xor)
print("XOR: final predictions  =", predict(X, w_xor, b_xor))

wrong_and = int(np.sum(predict(X, w_and, b_and) != y_and))
wrong_xor = int(np.sum(predict(X, w_xor, b_xor) != y_xor))

assert np.all(predict(X, w_and, b_and) == y_and)

print("Task 1 OK")

AND: epochs = 6 w = [1.  0.5] b = -1.0
AND: mistakes per epoch = [1, 3, 3, 2, 1, 0]
AND: final predictions  = [0 0 0 1]
XOR: epochs = 20 w = [-0.5  0. ] b = 0.5
XOR: mistakes per epoch = [2, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]
XOR: final predictions  = [1 1 0 0]
Task 1 OK


****
**Why must the parameters be reset before the XOR run?**

The weights hold what was learned on AND. Starting XOR from them would mix the two tasks, so it starts from w=0, b=0 to be an independent run

**Why does success on AND not imply success on XOR?**

A single perceptron can only draw one straight line. AND is linearly separable, so a line exists and the perceptron is guaranteed to find it. XOR is not linearly separable, so no such line exists

**Why can more epochs not overcome a single perceptron’s linear-boundary limitation on XOR?**

Training only moves the line and can't change the fact that a line is all the model has. Every (w, b) misclassifies at least one XOR point, so updates never stop (mistakes stay at 4 per epoch). Solving XOR needs a hidden layer or non-linear features

**Are online mistakes during an epoch always equal to the final model’s errors? Explain.**

No. Online mistakes are counted while the weights are changing, but final errors are measured with one fixed model afterward. On XOR the last epoch had 4 online mistakes, while the final model gets 2 wrong. They match only when an epoch has zero mistakes, as on AND (0 and 0), because then no updates occurred
****

#### Task 2

In [ ]:
x = np.array([0.5, 1.0, 1.5, 2.0])
y = np.array([2.0, 3.0, 4.0, 5.0])

params0 = np.array([0.5, 0.5, 1.0, 0.0])   # [w1, b1, w2, b2]


def forward(params, x):

    w1, b1, w2, b2 = params

    z1 = w1 * x + b1
    h = np.maximum(0.0, z1)
    yhat = w2 * h + b2
    return z1, h, yhat


def batch_loss(params, x, y):

    z1, h, yhat = forward(params, x)

    assert yhat.shape == y.shape

    return np.mean((yhat - y) ** 2)


def batch_gradients(params, x, y):

    w1, b1, w2, b2 = params

    z1, h, yhat = forward(params, x)

    N = len(x)

    g_yhat = (2.0 / N) * (yhat - y)          # per-example, shape (4,)
    g_w2 = g_yhat * h                        # per-example
    g_b2 = g_yhat                            # per-example
    g_h = g_yhat * w2                        # per-example
    g_z1 = g_h * (z1 > 0).astype(float)      # per-example
    g_w1 = g_z1 * x                          # per-example
    g_b1 = g_z1                              # per-example

    # shared parameters -> sum over examples
    return np.array([g_w1, g_b1, g_w2, g_b2])


z1, h, yhat = forward(params0, x)
print(f"{'x':>5}{'y':>6}{'z1':>7}{'h':>7}{'yhat':>7}{'sq.err':>9}")
for row in zip(x, y, z1, h, yhat, (yhat - y) ** 2):
    print("".join(f"{v:>{w}.4f}" for v, w in zip(row, [5, 6, 7, 7, 7, 9])))
print("yhat =", yhat)
print("mean loss =", batch_loss(params0, x, y))

g = batch_gradients(params0, x, y)
print("analytic grad [w1,b1,w2,b2] =", g)

# finite-difference check
eps = 1e-6
num = np.zeros(4)
for k in range(4):
    p1, p2 = params0.copy(), params0.copy()
    p1[k] += eps; p2[k] -= eps
    num[k] = (batch_loss(p1, x, y) - batch_loss(p2, x, y)) / (2 * eps)
print("numeric  grad               =", num)

assert np.allclose(g, num, atol=1e-6)

print("Task 2 OK")

    x     y     z1      h   yhat   sq.err
0.50002.0000 0.7500 0.7500 0.7500   1.5625
1.00003.0000 1.0000 1.0000 1.0000   4.0000
1.50004.0000 1.2500 1.2500 1.2500   7.5625
2.00005.0000 1.5000 1.5000 1.5000  12.2500
yhat = [0.75 1.   1.25 1.5 ]
mean loss = 6.34375
analytic grad [w1,b1,w2,b2] = [-6.875  -4.75   -5.8125 -4.75  ]
numeric  grad               = [-6.875  -4.75   -5.8125 -4.75  ]
Task 2 OK


****
**Where does the factor 1/N enter the gradients, and why?**
It enters once, at the start of the chain, in ∂L/∂ŷᵢ = (2/N)(ŷᵢ − yᵢ), because the loss is an average over N examples. Every later gradient is built from this term, so each one is automatically scaled by 1/N.

**Why do the shared-parameter gradients require a sum over the examples?**
The same four parameters (w₁, b₁, w₂, b₂) are used in all four examples. Changing one parameter changes every example's prediction, so its total effect on L is the sum of its effects through each example.

**What is the difference between the per-example vectors of shape (4,) and the returned parameter-gradient vector of shape (4,)?**
The per-example vectors (`g_yhat`, `g_z1`, `g_w1`, ...) have one entry per example (i = 1..4). The returned vector has one entry per parameter (w₁, b₁, w₂, b₂), and it is made by summing each per-example vector over the examples.

**Why is the output of this model not a probability?**
The output ŷ = w₂h + b₂ is linear and unbounded. It can be negative or greater than 1, and nothing forces it into [0, 1]. There is no sigmoid or softmax, and the model is trained with squared error toward real-valued targets (2 to 5), not class labels.
****

### Task 3

****
**Why do we perturb one parameter at a time, and what does .copy() prevent?**

Each gradient entry is the partial derivative with respect to one parameter, so only that coordinate can change while the other three stay fixed. .copy() prevents the perturbation from overwriting params and leaking into other coordinates.

**What does  control, and how does it differ from ?**

ε is the tiny step size used to probe the loss when estimating a derivative, and it never changes the parameters. η is the learning rate, the step size of the actual gradient-descent update, and it does change the parameters.

**Why must all gradients use the same parameter values before the update?**

The gradient is defined at one point in parameter space. If w2 were updated first, later gradients would be computed at a mixed point that is neither the old nor the new parameters, so the step would not follow the true gradient.

**Why does the loss history contain 201 values for 200 updates?**

One value is stored for the initial parameters (update 0), then one after each of the 200 updates
****

#### Part A

In [6]:
def numerical_gradient(fn, params, eps=1e-5):

    gradient = np.zeros_like(params, dtype=float)

    for j in range(len(params)):

        plus = params.copy()
        minus = params.copy()
        plus[j] += eps
        minus[j] -= eps
        gradient[j] = (fn(plus) - fn(minus)) / (2 * eps)

    return gradient

analytic = batch_gradients(params0, x, y)

numeric = numerical_gradient(
    lambda p: batch_loss(p, x, y), params0
)

print("analytic:", analytic)
print("numeric :", numeric)
print("max absolute difference:", np.max(np.abs(analytic - numeric)))

assert np.max(np.abs(analytic - numeric)) < 1e-6

print("Task 3A OK")

analytic: [-6.875  -4.75   -5.8125 -4.75  ]
numeric : [-6.875  -4.75   -5.8125 -4.75  ]
max absolute difference: 8.662937034387141e-11
Task 3A OK


#### Part B

In [7]:
params = params0.copy()
eta = 0.01


loss_history = [batch_loss(params, x, y)]

for step in range(200):

    gradients = batch_gradients(params, x, y)
    params = params - eta * gradients

    loss_history.append(batch_loss(params, x, y))


print("final params:", params)
print("final loss  :", loss_history[-1])

assert len(loss_history) == 201
assert loss_history[-1] < loss_history[0]

print("Task 3B OK")

final params: [1.21710839 0.65019193 1.54577988 0.1613669 ]
final loss  : 0.004726306469450738
Task 3B OK


****
**Which function performs backpropagation? Which statement updates the parameters?**

batch_gradients performs backpropagation. params = params - eta * gradients updates the parameters

**Why is updating a weight before finishing the backward pass incorrect?**

Later gradients depend on the current weights (for example, g_h = g_yhat * w2). Changing w2 early would make the remaining gradients use the wrong values.

**Does the reduced loss on these four training examples demonstrate performance on unseen data? Explain.**

No. It only shows the model fits these four points. Generalization has to be measured on held-out data the model never trained on, and with four points and four parameters the model could just be memorizing.
****